# Counterfactuals — CelebA (complex)

Prompt-to-Prompt counterfactual editing on CelebA with the four-attribute
complex SCM (Young / Male / No_Beard / Bald). Pipeline setup is shared via
`inference_utils.py`; this notebook owns the dataset loader, P2P sweep and
attention visualisation.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('/projects/dsai/se_aieng/cai/causal/workspace/Causal-Adapter/causal-adapter-sd15')

import os
import random
from copy import deepcopy

import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from torchvision.datasets import CelebA

from causal_modules.ddim_modules import (
    P2P_editing,
    sample,
    ddim_editing,
    save_images_grid,
)
from causal_modules.p2p_edits.attention_control import register_attention_control_controlnet
from inference_utils import build_transforms, load_causal_adapter

## 1. Configuration

In [ ]:
DATASET = 'celeA_complex'
SIZE = 256

BASE_MODEL_PATH = '/projects/dsai/se_aieng/cai/causal/workspace/causaledit/MCPL-diffuser/.cache/huggingface/hub/models--lambdalabs--miniSD-diffusers/snapshots/26ed8a9bfbf76f46a6cf60517dde321f900c44ce'

RUN_DIR = '/projects/dsai/se_aieng/cai/causal/workspace/causaledit/MCPL-diffuser/logs/logs_celeA_complex_all/2025-04-23T17-05-57-controlnet_textcond_constrastivegeneration_text_global_after'
STEPS = 200000
CONTROLNET_PATH     = f'{RUN_DIR}/controlnet-steps-{STEPS}.safetensors'
TEXT_EMBEDDING_PATH = f'{RUN_DIR}/learned_embeds-steps-{STEPS}.safetensors'
SCM_PATH            = '/projects/dsai/se_aieng/cai/causal/workspace/causaledit/MCPL-diffuser/logs/logs_celeA_complex_all/2025-04-23T21-11-19-causalnet_pretrain/best_model.pt'

DATA_ROOT = '/projects/dsai/se_aieng/cai/causal/workspace/causaledit/counterfactual-benchmark/datasets/'

## 2. Build pipeline + transforms

In [ ]:
image_transforms, original_transforms, _ = build_transforms(DATASET, size=SIZE)

assets = load_causal_adapter(
    DATASET,
    base_model_path=BASE_MODEL_PATH,
    controlnet_path=CONTROLNET_PATH,
    text_embedding_path=TEXT_EMBEDDING_PATH,
    scm_path=SCM_PATH,
)
pipe              = assets.pipe
controlnet        = assets.controlnet
device            = assets.device
prompt            = assets.prompt
presudo_words     = assets.presudo_words
presudo_list      = assets.presudo_list
presudo_token_ids = assets.presudo_token_ids

## 3. Load a CelebA sample

In [ ]:
def load_celeba_split(data_root=DATA_ROOT, dataset=controlnet.dataset, split='train'):
    data = CelebA(root=data_root, split=split, transform=None, download=False)
    if 'simple' in dataset:
        selected_attrs = ['Smiling', 'Eyeglasses']
    elif 'complex' in dataset:
        selected_attrs = ['Young', 'Male', 'No_Beard', 'Bald']
    else:
        raise AssertionError(f'no such {dataset} dataset')

    attribute_ids = [data.attr_names.index(a) for a in selected_attrs]
    attrs = torch.cat(
        [data.attr[:, i].float().unsqueeze(1) for i in attribute_ids], dim=1
    )
    return data, attrs, len(data)


data, imglabel, num_images = load_celeba_split()

img_id = 11  # set to ``random.randint(0, num_images - 1)`` for a fresh sample
img, label = data[img_id][0], imglabel[img_id].unsqueeze(0)
print('label:', label)
img

## 4. Prompt-to-Prompt counterfactual sweep

Per-attribute blend / replace hyper-parameters were tuned for the four
complex attributes; ``inter_value`` is the binary complement of the source
label (with an extra OOD case: switch hair if the subject is female).

In [ ]:
input_image = img.convert('RGB') if img.mode != 'RGB' else img
original_img = original_transforms(input_image.copy())
image_t = image_transforms(input_image)

inter_value = 1 - label.clone().squeeze()
if int(label.squeeze()[1].item()) == 0:  # female -> push the No_Beard axis OOD
    inter_value[2] = -1.0

set_guidance_scale = 3.0
num_steps = 50
invert_guidance_scale = 1.0
blend_word = True

BLEND_PRESETS = {
    0: dict(blend_params={'start_blend': 0.0, 'th': (0.3, 0.3)},
            cross_replace_steps=0.1, self_replace_steps=0.1),
    1: dict(blend_params={'start_blend': 0.0, 'th': (0.3, 0.5)},
            cross_replace_steps=0.2, self_replace_steps=0.2),
    2: dict(blend_params={'start_blend': 0.0, 'th': (0.3, 0.5)},
            cross_replace_steps=0.0, self_replace_steps=0.0),
    3: dict(blend_params={'start_blend': 0.0, 'th': (0.5, 0.3)},
            cross_replace_steps=0.0, self_replace_steps=0.0),
}

image_lists = [np.asarray(original_img)]
for inter_id in range(4):
    cfg = BLEND_PRESETS[inter_id]
    interved_image, inverted_latents, _, uncond_embeddings = P2P_editing(
        pipe, image_t.unsqueeze(0),
        label.clone(), presudo_token_ids, prompt, presudo_list,
        num_steps=num_steps,
        invert_guidance_scale=invert_guidance_scale,
        set_guidance_scale=set_guidance_scale,
        intervention_indx=inter_id,
        intervention_values=inter_value[inter_id],
        return_PIL=True,
        blend_word=blend_word,
        blend_params=cfg['blend_params'],
        disentangle=False,
        cross_replace_steps=cfg['cross_replace_steps'],
        self_replace_steps=cfg['self_replace_steps'],
        DSCM_labels=None,
    )
    image_lists.append(np.asarray(interved_image[-1]))

save_images_grid([image_lists], (1, len(image_lists)), './intervention_celeA.png')
print('saved ./intervention_celeA.png')

## 5. Traversal editing (single attribute, 30 steps)

Smoothly interpolate one attribute from its source value to the target
(``inter_value[k]``) via ``DSCM_labels`` while keeping the others fixed.

In [ ]:
TRAVERSE_ID = 1
n_traverse = 30

cfg = BLEND_PRESETS[TRAVERSE_ID]
label_bk = label.clone()
v_start = label_bk[0, TRAVERSE_ID].float()
v_end   = inter_value[TRAVERSE_ID].float()
vals = torch.linspace(v_start, v_end, steps=n_traverse).to(device)

image_lists = [np.asarray(original_img)]
for val in vals:
    DSCM_labels = label_bk.clone()
    DSCM_labels[:, TRAVERSE_ID] = val
    DSCM_labels = DSCM_labels.unsqueeze(2).to(device)

    interved_image, inverted_latents, _, uncond_embeddings = P2P_editing(
        pipe, image_t.unsqueeze(0),
        label.clone(), presudo_token_ids, prompt, presudo_list,
        num_steps=num_steps,
        invert_guidance_scale=invert_guidance_scale,
        set_guidance_scale=set_guidance_scale,
        intervention_indx=TRAVERSE_ID,
        intervention_values=val,
        return_PIL=True,
        blend_word=blend_word,
        blend_params=cfg['blend_params'],
        disentangle=True,
        cross_replace_steps=cfg['cross_replace_steps'],
        self_replace_steps=cfg['self_replace_steps'],
        DSCM_labels=DSCM_labels,
    )
    image_lists.append(np.asarray(interved_image[-1]))

save_path = f'./intervention_traverse_{TRAVERSE_ID}.png'
save_images_grid([image_lists], (1, len(image_lists)), save_path)
print(f'saved {save_path}')

## 6. Plain DDIM editing (no P2P)

Compares the P2P sweep against a vanilla DDIM-inversion + sample roundtrip.

In [ ]:
register_attention_control_controlnet(pipe, None)

image_re = img.convert('RGB') if img.mode != 'RGB' else img
original_img2 = original_transforms(image_re.copy())
image_t2 = image_transforms(image_re)

final_im, inverted_latents, _, uncond_embeddings = ddim_editing(
    pipe, image_t2.unsqueeze(0), label.clone(), presudo_token_ids, prompt,
    num_steps=num_steps,
    invert_guidance_scale=1.0,
    set_guidance_scale=1.0,
    intervention_indx=None,
    intervention_values=None,
    return_PIL=True,
    null_optimization=False,
)
save_images_grid([[original_img2, final_im[0]]], (1, 2), None)

In [ ]:
inter_value = 1 - label.clone().squeeze()
if int(label.squeeze()[1].item()) == 0:
    inter_value[2] = -1.0

image_lists = [np.asarray(original_img2)]
for inter_id in range(4):
    interved_image, _ = sample(
        pipe,
        prompt,
        presudo_token_ids,
        start_step=0,
        start_latents=inverted_latents[-1].clone(),
        guidance_scale=2.0,
        num_inference_steps=num_steps,
        num_images_per_prompt=1,
        negative_prompt=None,
        device=device,
        controlnet_image=None,
        intervention_indx=inter_id,
        intervention_values=inter_value[inter_id],
        label=label.clone(),
        return_PIL=True,
        disentangle=False,
        uncond_embeddings=None,
    )
    image_lists.append(np.asarray(interved_image[0]))

save_images_grid([image_lists], (1, len(image_lists)), './intervention_celeA_ddim.png')

## 7. Cross-attention maps

In [ ]:
from causal_modules.p2p_edits import ptp_tools, ptp_utils
import importlib
importlib.reload(ptp_tools)
importlib.reload(ptp_utils)

image_re = img.convert('RGB') if img.mode != 'RGB' else img
original_img3 = original_transforms(image_re.copy())
condition_image = image_re.copy()
image_t3 = image_transforms(image_re)

out_dir = os.path.join('./textcond', prompt)
os.makedirs(out_dir, exist_ok=True)

generator = torch.manual_seed(0)
overlapped_mask, attn_img = ptp_tools.plot_img_attn_mask_textcontrol(
    pipe, [prompt], presudo_words, presudo_token_ids, condition_image,
    device, out_dir, 'causalnet.png',
    latent=image_t3.unsqueeze(0), res=16, label=label,
    GUIDANCE_SCALE=1.0, attn_threshold=0.5, only_sampling=False,
    show_text=True, save_masks=False, class_select=False,
    intervention_indx=None, intervention_values=None,
    from_where=['down', 'up'], mask_concepts=True,
    g_gpu=generator, num_steps=30, img_size=SIZE,
    exp_names=['textcond'], dataset='celeba',
)